# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.
2. **Chạy trực tiếp (Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang.

In [34]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

Sat Nov  7 12:48:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [51]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os

# Luôn luôn xuất phát từ thư mục gốc /content (chống lỗi clone lồng nhau ZAIC_2026/ZAIC_2026)
%cd /content

if not os.path.exists('/content/ZAIC_2026/.git'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git /content/ZAIC_2026

%cd /content/ZAIC_2026
!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

/content
/content/ZAIC_2026
Already on 'planner/vuong'
Your branch is up to date with 'origin/planner/vuong'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.13 KiB | 579.00 KiB/s, done.
From https://github.com/Bamboohoccode/ZAIC_2026
 * branch            planner/vuong -> FETCH_HEAD
   8db22e2..0886107  planner/vuong -> origin/planner/vuong
Updating 8db22e2..0886107
Fast-forward
 notebooks/colab_run_planner.ipynb | 107 ++++++++++++++++++++++++++++----------
 1 file changed, 80 insertions(+), 27 deletions(-)

Commit mới nhất trên nhánh planner/vuong:
0886107 (HEAD -> planner/vuong, origin/planner/vuong) [notebooks] dong nhat gpu-mem thanh 0.90 chuan nhu Kaggle T4


In [37]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Tự động giải nén lên ổ cứng cục bộ /content)
import os, shutil, sys
from pathlib import Path

%cd /content/ZAIC_2026

# Kiểm tra xem /content/data có sẵn chưa
has_local_data = os.path.exists('/content/data/train/train.jsonl') and os.path.exists('/content/data/train/zaic_tools/apps/banking.py')

if not has_local_data:
    # Tìm file zip data.zip trên Drive hoặc /content
    zip_candidates = ['/content/drive/MyDrive/data.zip', '/content/data.zip', '/content/drive/MyDrive/ZAIC_2026/data.zip']
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if found_zip:
        print(f'📦 Đang giải nén dữ liệu từ {found_zip} sang ổ cứng cục bộ /content/ ... (mất ~15 giây)')
        !unzip -q -o "{found_zip}" -d /content/
    else:
        print('⚠️ Không tìm thấy data.zip trên Drive hoặc /content!')

# Tạo liên kết data/train và data/public_test trong repo
os.makedirs('data', exist_ok=True)
if os.path.exists('/content/data/train'):
    !rm -rf data/train
    !ln -s /content/data/train data/train
if os.path.exists('/content/data/public_test'):
    !rm -rf data/public_test
    !ln -s /content/data/public_test data/public_test

# KIỂM TRA TỰ ĐỘNG: Import thử zaic_tools để đảm bảo 100% không còn lỗi ModuleNotFoundError
try:
    if 'data/train' not in sys.path:
        sys.path.insert(0, 'data/train')
    from zaic_tools import ToolEnv
    print('✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!')
    !ls -lh data/train | head -n 6
except Exception as e:
    print('❌ LỖI DỮ LIỆU:', e)
    print('Vui lòng kiểm tra file data.zip có chứa thư mục zaic_tools/apps hay không!')

/content/ZAIC_2026
📦 Đang giải nén dữ liệu từ /content/drive/MyDrive/data.zip sang ổ cứng cục bộ /content/ ... (mất ~15 giây)
✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!
lrwxrwxrwx 1 root root 19 Nov  7 12:48 data/train -> /content/data/train


### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [52]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
%cd /content/ZAIC_2026
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.pipeline' || true
import os, time
time.sleep(2)
os.makedirs('runs', exist_ok=True)

# Chạy ngầm bằng nohup (đồng nhất chuẩn --gpu-mem 0.90 như Kaggle T4)
!nohup python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem 0.90 \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v4_guarded.jsonl > runs/dev_v4_guarded.nohup.log 2>&1 &

print('🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')

/content/ZAIC_2026
^C
🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!
Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.


In [39]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# %cd /content/ZAIC_2026
# import os
# os.makedirs('runs', exist_ok=True)
#
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem 0.90 \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v4_guarded.jsonl

In [ ]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Tự động cập nhật liên tục)
%cd /content/ZAIC_2026
import time, os
from IPython.display import clear_output

out_path = 'runs/dev_v4_guarded.jsonl'
log_path = 'runs/dev_v4_guarded.jsonl.log'
nohup_log = 'runs/dev_v4_guarded.nohup.log'

for loop in range(120):  # cập nhật mỗi 5 giây (kéo dài 10 phút, có thể chạy lại bất cứ lúc nào)
    clear_output(wait=True)
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    pids = os.popen("pgrep -f 'lifemate.pipeline'").read().strip().split()
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 65)
    
    # Tự động đồng bộ ngay sang Google Drive để bạn theo dõi trên điện thoại/Drive
    if os.path.exists('/content/drive/MyDrive') and done_count > 0:
        drive_dir = '/content/drive/MyDrive/ZAIC_2026_runs'
        os.makedirs(drive_dir, exist_ok=True)
        os.system(f'cp -f {out_path} {drive_dir}/ 2>/dev/null')
        os.system(f'cp -f {log_path} {drive_dir}/ 2>/dev/null')
    
    if os.path.exists(log_path) and os.path.getsize(log_path) > 0:
        !tail -n 12 runs/dev_v4_guarded.jsonl.log
    elif os.path.exists(nohup_log):
        print('⏳ Đang khởi động và làm nóng mô hình vLLM (Torch compile & CUDA graphs)...')
        print('Quá trình làm nóng này mất khoảng 2-3 phút trước khi chạy mẫu đầu tiên:')
        !tail -n 8 runs/dev_v4_guarded.nohup.log
    
    if pids:
        print(f'\n⚡ Tiến trình GPU đang xử lý (PID: {pids[0]})...')
        time.sleep(5)
    else:
        if done_count >= 300:
            print('\n🎉 TIẾN TRÌNH ĐÃ HOÀN THÀNH XONG ĐỦ 300 MẪU! Hãy chạy Cell 6 để chấm điểm.')
        elif done_count > 0:
            print(f'\n⏹️ Tiến trình đã dừng lại ở {done_count}/300 mẫu. Có thể chạy lại Cell 5A để tiếp tục.')
        else:
            print('\n⏹️ Chưa có tiến trình nào đang chạy.')
        break


📊 TIẾN ĐỘ: Đã xong 7/300 mẫu dev (2.3%)
[6/300] train_0038 running...
[6/300] train_0038 DONE calls=0 24.6s
[7/300] train_0050 running...
[7/300] train_0050 DONE calls=1 11.8s
[8/300] train_0072 running...
=== Lifemate Pipeline Run ===
Start: 2026-11-07T13:05:43.299321
Output: runs/dev_v4_guarded.jsonl | Trace: runs/dev_v4_guarded.jsonl.trace.jsonl | Log: runs/dev_v4_guarded.jsonl.log
Split: train | Total samples: 300
Resume active: 7/300 samples already finished in runs/dev_v4_guarded.jsonl
Prompt: prompts/gepa/vuong/planner_v4.txt
[8/300] train_0072 running...

⚡ Tiến trình GPU đang xử lý (PID: 12465)...


In [ ]:
!nvidia-smi

Sat Nov  7 13:03:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pkill -9 -f 'lifemate.pipeline'

In [41]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
%cd /content/ZAIC_2026
import os

sub_file = 'runs/dev_v4_guarded.jsonl'
if not os.path.exists(sub_file) or os.path.getsize(sub_file) == 0:
    print('⚠️ Chưa có dữ liệu kết quả để chấm điểm!')
    print('Lý do: Tiến trình GPU ở Cell 5 đang khởi động vLLM hoặc đang chạy những mẫu đầu tiên.')
    print('👉 Hãy đợi Cell 5C hoàn thành ít nhất 1 mẫu rồi chạy lại cell này để chấm điểm!')
else:
    with open(sub_file, 'r', encoding='utf-8') as f:
        n_samples = sum(1 for line in f if line.strip())
    print(f'Đang chấm điểm trên {n_samples}/300 mẫu đã hoàn thành...')
    !python eval/scorer.py --sub runs/dev_v4_guarded.jsonl


In [42]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
%cd /content/ZAIC_2026
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

/content/ZAIC_2026
Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: /content/drive/MyDrive/ZAIC_2026_runs
